# Lumen Travel · Ollama + Pinggy

Server cho Practice 03, dùng đúng `embeddinggemma` và `qwen3:4b`.

1. Chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU khả dụng).
2. Chạy các ô bên dưới theo thứ tự.
3. Chép URL và token từ ô cuối vào `.env` của ứng dụng. Không đưa token vào GitHub.
4. Giữ runtime hoạt động. Link tunnel miễn phí có thể hết hạn; khi đó chạy lại ô tunnel và cập nhật `.env`.

Notebook không dùng Firestore và không nhận cấu hình Firebase.


In [ ]:
import os, subprocess, time, json, secrets, re, sys, shutil
from pathlib import Path
from urllib.request import urlopen
from urllib.parse import urlsplit

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("Hãy chọn GPU trong Runtime → Change runtime type rồi chạy lại.")
print("GPU:", gpu.stdout.strip())
repo = Path("/content/lumen-travel-assistant")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vo-m-tri1412/lumen-travel-assistant.git", str(repo)], check=True)
# Colab images may omit zstd, which recent Ollama archives require.
if any(shutil.which(tool) is None for tool in ["curl", "zstd", "lspci", "lshw"]):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "curl", "ca-certificates", "zstd", "pciutils", "lshw"], check=True)
installation = subprocess.run(["bash", "-lc", "set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(installation.stdout[-6000:])
if installation.returncode != 0:
    raise RuntimeError("Ollama chưa cài được. Thông báo nguyên nhân nằm ngay phía trên; gửi phần thông báo đó để kiểm tra.")
env = {**os.environ, "OLLAMA_HOST":"127.0.0.1:11434", "OLLAMA_MAX_LOADED_MODELS":"1", "OLLAMA_NUM_PARALLEL":"1"}
try:
    urlopen("http://127.0.0.1:11434/api/tags", timeout=3).close()
except Exception:
    ollama_log = open("/tmp/lumen-ollama.log", "w")
    ollama_process = subprocess.Popen(["ollama", "serve"], env=env, stdout=ollama_log, stderr=subprocess.STDOUT)
    for attempt in range(60):
        try:
            urlopen("http://127.0.0.1:11434/api/tags", timeout=2).close()
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("Ollama chưa chạy. Xem /tmp/lumen-ollama.log.")
print("Ollama đã sẵn sàng.")


In [ ]:
for model in ["embeddinggemma", "qwen3:4b"]:
    print("Đang chuẩn bị", model, flush=True)
    subprocess.run(["ollama", "pull", model], env=env, check=True)
print("Đã có đủ hai model.")


In [ ]:
# Chạy lại ô này khi cần URL mới. Chỉ dừng các tiến trình do ô này tạo.
for name in ["proxy_process", "tunnel_process"]:
    previous = globals().get(name)
    if previous is not None and previous.poll() is None:
        previous.terminate()
        try:
            previous.wait(timeout=5)
        except subprocess.TimeoutExpired:
            previous.kill()
token = secrets.token_urlsafe(32)
proxy_log = open("/tmp/lumen-proxy.log", "w")
proxy_process = subprocess.Popen([sys.executable, str(repo/"server/proxy.py")], env={**os.environ,"OLLAMA_API_TOKEN":token}, stdout=proxy_log, stderr=subprocess.STDOUT)
time.sleep(1)
if proxy_process.poll() is not None:
    raise RuntimeError("Proxy chưa khởi động. Xem /tmp/lumen-proxy.log.")
tunnel_process = subprocess.Popen(["ssh", "-T", "-p", "443", "-o", "StrictHostKeyChecking=accept-new", "-o", "ServerAliveInterval=30", "-R", "0:localhost:11435", "a.pinggy.io"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
for line in tunnel_process.stdout:
    for candidate in re.findall(r"https://[A-Za-z0-9.-]+", line):
        host = urlsplit(candidate).hostname or ""
        if host.endswith((".pinggy.link", ".pinggy.io")):
            url = candidate
            break
    if url:
        break
if not url:
    raise RuntimeError("Chưa nhận được link Pinggy. Chạy lại ô này.")
print("Chép hai dòng sau vào .env trên máy chạy Streamlit:")
print("OLLAMA_BASE_URL=" + url)
print("OLLAMA_API_TOKEN=" + token)
print("Giữ runtime này hoạt động. Token chỉ dành cho phiên server này.")
